In [1]:
import os

ucl_path = r"C:\Users\Asus\OneDrive\Desktop\neopain-research\data\raw\UCL"

for root, dirs, files in os.walk(ucl_path):
    level = root.replace(ucl_path, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}📁 {os.path.basename(root)}/")
    sub_indent = '  ' * (level + 1)
    for f in sorted(files)[:10]:  # first 10 files per folder
        size_kb = os.path.getsize(os.path.join(root, f)) // 1024
        print(f"{sub_indent}📄 {f}  ({size_kb} KB)")
    if len(files) > 10:
        print(f"{sub_indent}... and {len(files)-10} more files")

📁 UCL/
  📁 Database/
    📄 (1) Infant Demographics.xlsx  (68 KB)
    📄 (2) Study details.xlsx  (31 KB)
    📄 (3) Stimulation information.xlsx  (65 KB)
    📄 (4) Infant patient notes.xlsx  (189 KB)
    📄 (5) Maternal patient notes.xlsx  (18 KB)
  📁 EEG/
    📁 2510001/
      📄 2510001A01.mat  (596 KB)
      📄 2510001C01.mat  (596 KB)
      📄 2510001L01.mat  (7374 KB)
    📁 2510101/
      📄 2510101A01.mat  (625 KB)
      📄 2510101C01.mat  (625 KB)
      📄 2510101L01.mat  (7818 KB)
    📁 2510201/
      📄 2510201A01.mat  (605 KB)
      📄 2510201C01.mat  (609 KB)
      📄 2510201L01.mat  (7765 KB)
    📁 2510301/
      📄 2510301A01.mat  (625 KB)
      📄 2510301C01.mat  (626 KB)
      📄 2510301L01.mat  (7202 KB)
    📁 2510401/
      📄 2510401A01.mat  (627 KB)
      📄 2510401C01.mat  (631 KB)
      📄 2510401L01.mat  (7717 KB)
    📁 2510501/
      📄 2510501A01.mat  (546 KB)
      📄 2510501C01.mat  (550 KB)
      📄 2510501L01.mat  (6750 KB)
    📁 2510601/
      📄 2510601A01.mat  (643 KB)
      📄 2

In [2]:
import scipy.io
import numpy as np

ucl_eeg_path = r"C:\Users\Asus\OneDrive\Desktop\neopain-research\data\raw\UCL\EEG"

# Load one heel lance file
mat = scipy.io.loadmat(f"{ucl_eeg_path}\\2510001\\2510001L01.mat")

# See all variables inside
print("Keys in .mat file:")
for key, val in mat.items():
    if not key.startswith('_'):
        if hasattr(val, 'shape'):
            print(f"  {key}: shape={val.shape}, dtype={val.dtype}")
        else:
            print(f"  {key}: {type(val)} = {val}")

Keys in .mat file:
  EEG: shape=(1, 1), dtype=[('setname', 'O'), ('filename', 'O'), ('filepath', 'O'), ('subject', 'O'), ('condition', 'O'), ('session', 'O'), ('nbchan', 'O'), ('trials', 'O'), ('pnts', 'O'), ('srate', 'O'), ('xmin', 'O'), ('xmax', 'O'), ('times', 'O'), ('data', 'O'), ('icaact', 'O'), ('icawinv', 'O'), ('icasphere', 'O'), ('icaweights', 'O'), ('icachansind', 'O'), ('chanlocs', 'O'), ('urchanlocs', 'O'), ('chaninfo', 'O'), ('ref', 'O'), ('event', 'O'), ('urevent', 'O'), ('eventdescription', 'O'), ('epoch', 'O'), ('epochdescription', 'O'), ('reject', 'O'), ('stats', 'O'), ('specdata', 'O'), ('specicaact', 'O'), ('splinefile', 'O'), ('icasplinefile', 'O'), ('dipfit', 'O'), ('history', 'O'), ('saved', 'O'), ('etc', 'O'), ('other_data', 'O')]


In [3]:
import scipy.io
import numpy as np
import pandas as pd
import os

ucl_eeg_path = r"C:\Users\Asus\OneDrive\Desktop\neopain-research\data\raw\UCL\EEG"
db_path      = r"C:\Users\Asus\OneDrive\Desktop\neopain-research\data\raw\UCL\Database"

# ── STEP 1: Correctly load one .mat file ─────────────────────────────
mat = scipy.io.loadmat(
    f"{ucl_eeg_path}\\2510001\\2510001L01.mat",
    squeeze_me=True,
    struct_as_record=False
)
eeg = mat['EEG']

print("=" * 50)
print("SINGLE .MAT FILE INSPECTION")
print("=" * 50)
print(f"Subject       : {eeg.subject}")
print(f"Condition     : {eeg.condition}")
print(f"Sampling rate : {eeg.srate} Hz")
print(f"Channels      : {eeg.nbchan}")
print(f"Time points   : {eeg.pnts}")
print(f"Duration      : {eeg.pnts / eeg.srate:.1f} seconds")
print(f"EEG data shape: {eeg.data.shape}")
print(f"Time axis     : {eeg.times[0]:.3f}s to {eeg.times[-1]:.3f}s")

# Channel names
try:
    ch_names = [str(eeg.chanlocs[i].labels) for i in range(len(eeg.chanlocs))]
    print(f"Channel names : {ch_names}")
except Exception as ex:
    print(f"Channel names : could not parse — {ex}")

# other_data — HR and SpO2 pre-computed values
print(f"\nother_data type  : {type(eeg.other_data)}")
try:
    od = eeg.other_data
    if hasattr(od, 'dtype') and od.dtype.names:
        print(f"other_data fields: {od.dtype.names}")
        for name in od.dtype.names:
            print(f"  {name}: {od[name]}")
    else:
        print(f"other_data shape : {od.shape}")
        print(f"other_data value : {od}")
except Exception as ex:
    print(f"other_data error : {ex}")


SINGLE .MAT FILE INSPECTION
Subject       : 25100
Condition     : lance
Sampling rate : 2000 Hz
Channels      : 18
Time points   : 8000
Duration      : 4.0 seconds
EEG data shape: (18, 8000)
Time axis     : -2000.000s to 1999.500s
Channel names : ['Cz', 'CPz', 'F8', 'T8', 'TP10', 'P8', 'O2', 'F4', 'C4', 'CP4', 'F7', 'T7', 'TP9', 'P7', 'O1', 'F3', 'C3', 'CP3']

other_data type  : <class 'scipy.io.matlab._mio5_params.mat_struct'>
other_data error : 'mat_struct' object has no attribute 'shape'


In [4]:
import pandas as pd

db_path = r"C:\Users\Asus\OneDrive\Desktop\neopain-research\data\raw\UCL\Database"

# File 1 — all sheets
xl1 = pd.ExcelFile(f"{db_path}\\(1) Infant Demographics.xlsx")
print("File 1 sheets:", xl1.sheet_names)

# File 2 — all sheets
xl2 = pd.ExcelFile(f"{db_path}\\(2) Study details.xlsx")
print("File 2 sheets:", xl2.sheet_names)

# File 3 — all sheets (3 tabs)
xl3 = pd.ExcelFile(f"{db_path}\\(3) Stimulation information.xlsx")
print("File 3 sheets:", xl3.sheet_names)

# File 4 — all sheets
xl4 = pd.ExcelFile(f"{db_path}\\(4) Infant patient notes.xlsx")
print("File 4 sheets:", xl4.sheet_names)

# File 5 — all sheets
xl5 = pd.ExcelFile(f"{db_path}\\(5) Maternal patient notes.xlsx")
print("File 5 sheets:", xl5.sheet_names)

File 1 sheets: ['Demographics', 'Delivery details', 'SNAP scores']
File 2 sheets: ['Study context', 'EEG details']
File 3 sheets: ['Heel lance', 'Sham control', 'Auditory control']
File 4 sheets: ['Ventilation', 'Diagnosis', 'Cranial scans', 'Medication', 'Heel lances', 'Painful procedures', 'Injuries']
File 5 sheets: ['Maternal']
